In [ ]:
import numpy as np
import pandas as pd

import pymcdm 

Helpers

In [ ]:
alpha = 0.88    # Sensitivity to gains
beta = 0.88     # Sensitivity to losses
lambda_ = 2.25  # Loss aversion parameter

def prospect_theory_value_matrix(value, neutral_value, criteria_type, alpha=alpha, beta=beta, lambda_=lambda_):
    diff = value - neutral_value

    if criteria_type == 1:
        if diff > 0:    
            return (value - neutral_value)**alpha
        return -lambda_ * (-(value - neutral_value)) ** beta
    
    if diff < 0:    
        return (-(value - neutral_value))**alpha
    return -lambda_ * ((value - neutral_value)) ** beta

def prospect_normalize_matrix(X):

    X = np.array(X, dtype=float)
    X_norm = np.zeros_like(X)

    for j in range(X.shape[1]):

        col = X[:, j]

        pos_vals = col[col > 0]
        neg_vals = col[col < 0]

        max_gain = np.max(pos_vals) if pos_vals.size > 0 else None
        max_loss = abs(np.min(neg_vals)) if neg_vals.size > 0 else None

        for i, val in enumerate(col):

            if val > 0 and max_gain is not None:
                X_norm[i, j] = val / max_gain

            elif val < 0 and max_loss is not None:
                X_norm[i, j] = val / max_loss

            else:
                X_norm[i, j] = 0

    return X_norm

def max_abs_normalization(x):
    return x / np.max(np.abs(x), axis=0)


In [ ]:
def ridm(
    matrix, 
    neutral_values, 
    criteria_types, 
    weights, 
    alpha_: float = 0.88,
    beta_: float = 0.88,
    lambda_: float = 2.25, 
    verbose: bool = False,
    normalization: callable = None
):
    if normalization and normalization not in [max_abs_normalization,prospect_normalize_matrix]:
        nmatrix = normalization(matrix)
        np_matrix = np.vstack([matrix, neutral_values])
        n_neutral_values = normalization(np_matrix)[-1, :]
    else: 
        nmatrix= matrix.copy()
        n_neutral_values = neutral_values.copy()

    result_matrix = np.zeros_like(nmatrix).astype(float)
    result_matrix_neg = np.zeros_like(nmatrix).astype(float)


    for i in range(nmatrix.shape[1]):    
        for k in range(nmatrix.shape[0]):
            result_matrix_neg[k, i] = prospect_theory_value_matrix(
                nmatrix[k, i], 
                n_neutral_values[i], 
                criteria_types[i],
                alpha=alpha_,
                beta=beta_,
                lambda_=lambda_
            )

    if normalization and normalization in [max_abs_normalization,prospect_normalize_matrix]:
        result_matrix_neg = normalization(result_matrix_neg)
         
    wmatrix = weights * result_matrix_neg

    ri = np.sum(wmatrix, axis=1)
    rank = pymcdm.helpers.rrankdata(ri)

    if verbose:
        print(np.round(result_matrix_neg, 3))
        print(np.round(wmatrix, 3))
        print(ri)
        print(rank)

    return ri, rank


RIDM extension for local weights

In [ ]:
def ridm_local_weights(
    matrix, 
    neutral_values, 
    criteria_types, 
    weights, 
    alpha_: float = 0.88,
    beta_: float = 0.88,
    lambda_: float = 2.25, 
    verbose: bool = False,
    normalization: callable = None,
    with_weights: bool = True
):
    if normalization and normalization not in [max_abs_normalization,prospect_normalize_matrix]:
        nmatrix = normalization(matrix)
        np_matrix = np.vstack([matrix, neutral_values])
        n_neutral_values = normalization(np_matrix)[-1, :]
    else: 
        nmatrix= matrix.copy()
        n_neutral_values = neutral_values.copy()

    result_matrix = np.zeros_like(nmatrix).astype(float)
    result_matrix_neg = np.zeros_like(nmatrix).astype(float)


    for i in range(nmatrix.shape[1]):    
        for k in range(nmatrix.shape[0]):
            result_matrix_neg[k, i] = prospect_theory_value_matrix(
                nmatrix[k, i], 
                n_neutral_values[i], 
                criteria_types[i],
                alpha=alpha_,
                beta=beta_,
                lambda_=lambda_
            )

    if normalization and normalization in [max_abs_normalization,prospect_normalize_matrix]:
        result_matrix_neg = normalization(result_matrix_neg)

    if with_weights:
        wmatrix = weights * result_matrix_neg
    else:
        wmatrix = result_matrix_neg

    matrix_weights = np.abs(wmatrix) / np.sum(np.abs(wmatrix), axis=0)
    row_sums = np.sum(np.abs(wmatrix), axis=1)

    local_weights = np.abs(wmatrix) / row_sums[:, np.newaxis]

    if verbose:
        print('Normalized behavioral matrix')
        print(np.round(result_matrix_neg, 3))
        
    return local_weights


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

def vis_heatmap(local_weights, vmin, vmax, title='', filename= None):
    alternatives = ["A1", "A2", "A3", "A4"]
    criteria = ["C1", 'C2', 'C3', 'C4', 'C5', 'C6', 'C7']

    df = pd.DataFrame(local_weights, index=alternatives, columns=criteria)

    plt.figure(figsize=(8, 5))
    sns.set(style="white")

    ax = sns.heatmap(
        df,
        annot=True,           
        fmt=".2f",
        cmap="Reds",      
        vmin=vmin,
        vmax=vmax,
        linewidths=2,
        cbar_kws={"label": "Local Weights"},
        annot_kws={'fontsize': 14}
    )

    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=12)
    cbar.set_label("Local weights", size=14)

    plt.title(title, fontsize=14)
    plt.xlabel("Criteria", fontsize=14)
    plt.ylabel("Alternatives", fontsize=14)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)

    plt.tight_layout()
    if filename:
        plt.savefig(f'img/{filename}.png')
    plt.show()

Illustrative example

In [ ]:
matrix = np.array([
    [20.0, 85.0, 75.0, 12.0, 65.0, 30.0, 40.0],  # A1
    [45.0, 60.0, 40.0, 5.5,  85.0, 65.0, 75.0],  # A2
    [70.0, 45.0, 25.0, 2.0,  72.0, 85.0, 90.0],  # A3
    [95.0, 68.0, 5.0,  0.5,  90.0, 90.0, 95.0]   # A4
])
neutral_values = np.array([30.0, 70.0, 50.0, 15.0, 60.0, 50.0, 50.0])

criteria_types = np.array([-1, -1, -1, -1, 1, 1, 1])

weights = pymcdm.weights.equal_weights(matrix)

In [ ]:
profile_base = {
    "name": "Classical Prospect",
    "alpha": 0.88,
    "beta": 0.88,
    "lambda": 2.25
}

profile_loss_averse = {
    "name": "Strong Loss-Averse",
    "alpha": 0.88,
    "beta": 0.95,
    "lambda": 3.5
}

profile_gain_seeking = {
    "name": "Gain-Oriented",
    "alpha": 0.95,
    "beta": 0.80,
    "lambda": 1.2
}

profile_neutral = {
    "name": "Near Rational",
    "alpha": 1.0,
    "beta": 1.0,
    "lambda": 1.0
}

In [ ]:
all_lw = []

dms = [
    profile_base,
    profile_loss_averse,
    profile_gain_seeking,
    profile_neutral
]

for dm in dms:
    print(dm['name'])
    lw = ridm_local_weights(
        matrix, 
        neutral_values, 
        criteria_types, 
        weights, 
        alpha_=dm['alpha'],
        beta_=dm['beta'],
        lambda_=dm['lambda'],
        normalization=max_abs_normalization,
        verbose=True
    )
    print('Local weights')
    print(lw)
    print('')
    all_lw.append(lw)


global_min = min(lw.min() for lw in all_lw)
global_max = max(lw.max() for lw in all_lw)

for idx, lw in enumerate(all_lw):
    title = fr'$\alpha$ = {dms[idx]["alpha"]:.2f}    $\beta$ = {dms[idx]["beta"]:.2f}   $\lambda$ = {dms[idx]["lambda"]:.2f}'
    vis_heatmap(lw, global_min, global_max, title, filename=f'dm_{idx}')

In [ ]:
import numpy as np

def get_cell_style(value, neutral, crit_type):
    if crit_type == 1:  # max
        if value >= neutral:
            return 'green', '↑'
        else:
            return 'red', '↓'
    else:  # min
        if value <= neutral:
            return 'green', '↓'
        else:
            return 'red', '↑'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from matplotlib.colors import to_rgb
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

def get_cell_style(value, neutral, crit_type):
    if value == 0.0:
        return 'white', '-'
    if crit_type == 1:  # max
        if value >= neutral:
            return 'green', '↑'
        else:
            return 'red', '↓'
    else:  # min
        if value <= neutral:
            return 'green', '↓'
        else:
            return 'red', '↑'


def blend_with_white(color, intensity):
    """
    intensity: 0 → white, 1 → full color
    """
    white = np.array([1, 1, 1])
    base = np.array(to_rgb(color))
    return white * (1 - intensity) + base * intensity


def vis_heatmap_dual(local_weights, matrix, neutral_values, criteria_types, vmax, ax, idx, title='', cbar_on=False):
    alternatives = ["$A_1$", "$A_2$", "$A_3$", "$A_4$"]
    criteria = ["$C_1$", '$C_2$', '$C_3$', '$C_4$', '$C_5$', '$C_6$', '$C_7$']

    df = pd.DataFrame(local_weights, index=alternatives, columns=criteria)

    annotations = []
    color_image = np.zeros((df.shape[0], df.shape[1], 3))

    for i in range(df.shape[0]):
        annot_row = []
        for j in range(df.shape[1]):
            val = matrix[i, j]
            
            neutral = neutral_values[j]
            ctype = criteria_types[j]

            lw = df.iloc[i, j]

            if lw == 0:
                color = 'white'
                arrow = '-'
            else:
                color, arrow = get_cell_style(val, neutral, ctype)

            intensity = abs(lw) / vmax if vmax != 0 else 0
            intensity = min(intensity, 1)

            if color == 'green':
                rgb = blend_with_white("#2ca02c", intensity)
            else:
                rgb = blend_with_white("#d62728", intensity)

            color_image[i, j] = rgb

            annot_row.append(f"{lw:.2f}")

        annotations.append(annot_row)

    annotations = np.array(annotations)

    ax.imshow(color_image, aspect='auto')

    for i in range(df.shape[0]):
        for j in range(df.shape[1]):
            ax.text(j, i, annotations[i, j], ha='center', va='center', fontsize=18, color='black')

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_xticks(np.arange(len(criteria)))
    ax.set_yticks(np.arange(len(alternatives)))

    ax.set_xticklabels(criteria, fontsize=18)
    ax.set_yticklabels(alternatives, fontsize=18)

    ax.set_xticks(np.arange(-0.5, len(criteria), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(alternatives), 1), minor=True)

    ax.grid(which='minor', color='white', linestyle='-', linewidth=2)
    ax.tick_params(which='minor', bottom=False, left=False)
    
    ax.set_title(title, fontsize=18)
    if idx > 1:
        ax.set_xlabel("Criteria", fontsize=20)
    if not idx % 2:
        ax.set_ylabel("Alternatives", fontsize=20)

    # Colorbar: Loss – Neutral – Gain
    if cbar_on:
        cmap = LinearSegmentedColormap.from_list(
            "loss_neutral_gain",
            ["#d62728", "white", "#2ca02c"]
        )

        norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])

        cbar = plt.colorbar(sm, ax=ax, fraction=0.035, pad=0.04)

        cbar.set_ticks([-vmax, 0, vmax])
        cbar.set_ticklabels(["Loss", "Neutral", "Gain"])
        cbar.ax.tick_params(labelsize=14, length=0)
        cbar.ax.set_title("Perceived impact", fontsize=16, pad=12)

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(7, 5), sharex=True, sharey=True)
global_max = max(np.abs(lw).max() for lw in all_lw[0:1])

vis_heatmap_dual(
    all_lw[0],
    matrix,
    neutral_values,
    criteria_types,
    ax=axes,
    vmax=global_max,
    idx=idx,
    cbar_on=True
)

axes.set_ylabel("Alternatives", fontsize=18)
plt.tight_layout()
plt.savefig("img/lw.png")
plt.show()

In [ ]:
from matplotlib.cm import ScalarMappable


fig = plt.figure(figsize=(15, 10))
gs = fig.add_gridspec(
    2, 3,
    width_ratios=[1, 1, 0.045],
    wspace=0.15,
    hspace=0.3
)

axes = [
    fig.add_subplot(gs[0, 0]),
    fig.add_subplot(gs[0, 1]),
    fig.add_subplot(gs[1, 0]),
    fig.add_subplot(gs[1, 1])
]

cax = fig.add_subplot(gs[:, 2])

global_max = max(np.abs(lw).max() for lw in all_lw)
labels = ['(A)', '(B)', '(C)', '(D)']

for idx, lw in enumerate(all_lw):
    title = fr'{labels[idx]} $\alpha$={dms[idx]["alpha"]:.2f}  $\beta$={dms[idx]["beta"]:.2f}  $\lambda$={dms[idx]["lambda"]:.2f}'

    vis_heatmap_dual(
        lw,
        matrix,
        neutral_values,
        criteria_types,
        vmax=global_max,
        ax=axes[idx],
        title=title,
        idx=idx
    )

cmap = LinearSegmentedColormap.from_list(
    "loss_neutral_gain",
    ["#d62728", "white", "#2ca02c"]
)

norm = TwoSlopeNorm(vmin=-global_max, vcenter=0, vmax=global_max)

sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

cbar = fig.colorbar(sm, cax=cax)

cbar.set_ticks([-global_max, 0, global_max])
cbar.set_ticklabels(["Loss", "Neutral", "Gain"])
cbar.ax.tick_params(labelsize=16, length=0)
cbar.ax.set_title("Perceived impact", fontsize=16, pad=12)

plt.tight_layout()
plt.savefig("img/all_dm_dual.pdf", dpi=300)
plt.show()

Different neutral reference points

In [ ]:
all_lw = []

neutral_status_quo_baseline = np.array([30.0, 70.0, 50.0, 15.0, 60.0, 50.0, 50.0])
neutral_status_quo_green = np.array([60.0, 45.0, 20.0, 5.0, 80.0, 75.0, 80.0])
neutral_status_quo_austerity = np.array([15.0, 85.0, 70.0, 25.0, 50.0, 30.0, 35.0])
neutral_status_quo_regulatory = np.array([40.0, 60.0, 15.0, 8.0, 65.0, 60.0, 70.0])

dms = [
    neutral_status_quo_baseline,
    neutral_status_quo_green,
    neutral_status_quo_austerity,
    neutral_status_quo_regulatory
]

for dm in dms:
    print(dm)
    lw = ridm_local_weights(
        matrix, 
        dm, 
        criteria_types, 
        weights, 
        alpha_=0.88,
        beta_=0.88,
        lambda_=2.25,
        normalization=max_abs_normalization,
        verbose=True
    )
    print('Local weights')
    print(lw)
    print('')
    all_lw.append(lw)


global_min = min(lw.min() for lw in all_lw)
global_max = max(lw.max() for lw in all_lw)

for idx, lw in enumerate(all_lw):
    title = ''
    vis_heatmap(lw, global_min, global_max, title, filename=f'refs_{idx}')

In [ ]:
fig = plt.figure(figsize=(15, 10))
gs = fig.add_gridspec(
    2, 3,
    width_ratios=[1, 1, 0.045],
    wspace=0.15,
    hspace=0.3
)

axes = [
    fig.add_subplot(gs[0, 0]),
    fig.add_subplot(gs[0, 1]),
    fig.add_subplot(gs[1, 0]),
    fig.add_subplot(gs[1, 1])
]

cax = fig.add_subplot(gs[:, 2])


global_max = max(np.abs(lw).max() for lw in all_lw)
labels = ['(P1) Baseline reference points', '(P2) Green transition perspective', '(P3) Austerity perspective', '(P4) Regulatory perspective']

for idx, lw in enumerate(all_lw):
    title = labels[idx]

    vis_heatmap_dual(
        lw,
        matrix,
        dms[idx],
        criteria_types,
        vmax=global_max,
        ax=axes[idx],
        title=title,
        idx=idx
    )

cmap = LinearSegmentedColormap.from_list(
    "loss_neutral_gain",
    ["#d62728", "white", "#2ca02c"]
)

norm = TwoSlopeNorm(vmin=-global_max, vcenter=0, vmax=global_max)

sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

cbar = fig.colorbar(sm, cax=cax)

cbar.set_ticks([-global_max, 0, global_max])
cbar.set_ticklabels(["Loss", "Neutral", "Gain"])
cbar.ax.tick_params(labelsize=16, length=0)
cbar.ax.set_title("Perceived impact", fontsize=16, pad=12)

plt.tight_layout()
plt.savefig("img/all_refs_dual.pdf", dpi=300)
plt.show()